In [1]:
!apt-get update
!apt-get install ghostscript -y

!pip install "camelot-py[cv]" opencv-python-headless

Hit:1 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://cli.github.com/packages stable InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ghostscript is already the newest version (9.55.0~dfsg1-0ubuntu5.13).
0 upgraded, 0 newly inst

In [2]:
import os
import requests
import urllib.parse
from bs4 import BeautifulSoup
from urllib.parse import urljoin

BASE_URL = "https://www.fia.com"

# The specific 2025 Season URL you provided
SEASON_BASE_URL = "https://www.fia.com/documents/championships/fia-formula-one-world-championship-14/season/season-2025-2071"

# The specific races you want to scrape right now
RACES = [
    "Abu Dhabi Grand Prix",
    "Australian Grand Prix",
    "Austrian Grand Prix",
    "Azerbaijan Grand Prix",
    "Bahrain Grand Prix",
    "Belgian Grand Prix",
    "British Grand Prix",
    "Canadian Grand Prix",
    "Chinese Grand Prix",
    "Dutch Grand Prix",
    "Emilia Romagna Grand Prix",
    "Hungarian Grand Prix",
    "Italian Grand Prix",
    "Japanese Grand Prix",
    "Las Vegas Grand Prix",
    "Mexico City Grand Prix",
    "Miami Grand Prix",
    "Monaco Grand Prix",
    "Qatar Grand Prix",
    "São Paulo Grand Prix",
    "Singapore Grand Prix",
    "Spanish Grand Prix",
    "United States Grand Prix"
]

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

TARGET_KEYWORDS = ["car presentation", "automobile display"]

def setup_download_folder(folder_name="fia_pdfs"):
    """Creates a local directory to save the downloaded PDFs."""
    if not os.path.exists(folder_name):
        os.makedirs(folder_name)
        print(f"Created folder: {folder_name}")
    return folder_name

def get_pdf_links(url, race_name):
    """Scrapes the FIA page and returns a list of URLs for matching PDFs."""
    print(f"\nScanning {race_name} for upgrade documents...")

    try:
        response = requests.get(url, headers=HEADERS)
        response.raise_for_status()
    except requests.exceptions.RequestException as e:
        print(f"  -> Failed to connect to the FIA website: {e}")
        return []

    soup = BeautifulSoup(response.content, 'html.parser')
    found_links = []

    for row in soup.find_all('li', class_='document-row'):
        title_element = row.find(class_='title')
        link_element = row.find('a')

        if title_element and link_element:
            text = title_element.get_text(strip=True).lower()
            href = link_element.get('href')

            if any(keyword in text for keyword in TARGET_KEYWORDS):
                full_pdf_url = urljoin(BASE_URL, href)

                safe_race_name = race_name.replace(" ", "_")
                filename = f"{safe_race_name}_{text}"

                found_links.append({"title": filename, "url": full_pdf_url})

    return found_links

def download_pdfs(pdf_list, download_folder):
    """Downloads each PDF in the list to the specified folder."""
    if not pdf_list:
        print("  -> No matching documents found on this page.")
        return

    print(f"  -> Found {len(pdf_list)} document(s)! Starting download...")

    for item in pdf_list:
        title = item['title'].replace(" ", "_").replace("/", "-") # Clean up title for filename
        pdf_url = item['url']

        if not title.endswith(".pdf"):
            title += ".pdf"

        file_path = os.path.join(download_folder, title)

        if os.path.exists(file_path):
            print(f"  -> Already exists: {title}")
            continue

        print(f"  -> Downloading: {title}...")

        try:
            doc_response = requests.get(pdf_url, headers=HEADERS, stream=True)
            doc_response.raise_for_status()

            with open(file_path, 'wb') as f:
                for chunk in doc_response.iter_content(chunk_size=8192):
                    f.write(chunk)
            print("  -> Success!")

        except Exception as e:
            print(f"  -> Failed to download {title}: {e}")

if __name__ == "__main__":
    save_dir = setup_download_folder()

    for race in RACES:
        encoded_race = urllib.parse.quote(race)

        race_url = f"{SEASON_BASE_URL}/event/{encoded_race}"

        target_pdfs = get_pdf_links(race_url, race)

        download_pdfs(target_pdfs, save_dir)

    print("\nScraping process complete!")


Scanning Abu Dhabi Grand Prix for upgrade documents...
  -> Found 1 document(s)! Starting download...
  -> Already exists: Abu_Dhabi_Grand_Prix_doc_8_-_car_presentation_submissions.pdf

Scanning Australian Grand Prix for upgrade documents...
  -> Found 1 document(s)! Starting download...
  -> Already exists: Australian_Grand_Prix_doc_7_-_car_presentation_submissions.pdf

Scanning Austrian Grand Prix for upgrade documents...
  -> Found 1 document(s)! Starting download...
  -> Already exists: Austrian_Grand_Prix_doc_9_-_car_presentation_submissions.pdf

Scanning Azerbaijan Grand Prix for upgrade documents...
  -> Found 1 document(s)! Starting download...
  -> Already exists: Azerbaijan_Grand_Prix_doc_8_-_car_presentation_submissions.pdf

Scanning Bahrain Grand Prix for upgrade documents...
  -> Found 1 document(s)! Starting download...
  -> Already exists: Bahrain_Grand_Prix_doc_10_-_car_presentation_submissions.pdf

Scanning Belgian Grand Prix for upgrade documents...
  -> Found 1 docu

In [3]:
!pip install pdfplumber

In [4]:
import os
import camelot
import pdfplumber
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

STOP_WORDS = set(stopwords.words('english'))

CONSTRUCTORS = {
    "MCLAREN": "McLaren",
    "FERRARI": "Ferrari",
    "RED BULL": "Red Bull Racing",
    "MERCEDES": "Mercedes",
    "ASTON MARTIN": "Aston Martin",
    "ALPINE": "Alpine",
    "HAAS": "Haas",
    "RACING BULLS": "Visa Cash App RB",
    "WILLIAMS": "Williams",
    "SAUBER": "Kick Sauber"
}

def clean_text_stopwords(text):
    """
    Tokenizes the text, removes English stopwords, and rejoins the string.
    """
    if not text:
        return text

    words = word_tokenize(text)
    filtered_words = [word for word in words if word.lower() not in STOP_WORDS]
    return " ".join(filtered_words)

def extract_race_name(filename):
    clean_name = filename.replace(".pdf", "")
    if "_doc_" in clean_name:
        clean_name = clean_name.split("_doc_")[0]
    if " - " in clean_name:
        clean_name = clean_name.split(" - ")[0]

    clean_name = clean_name.replace("_", " ").strip()
    return re.sub(r'^\d{4}\s+', '', clean_name)

def get_team_per_page(file_path):
    page_team_map = {}
    with pdfplumber.open(file_path) as pdf:
        for idx, page in enumerate(pdf.pages):
            page_num = idx + 1
            page_text = page.extract_text() or ""

            current_team = "Unknown Team"
            for key, formal_name in CONSTRUCTORS.items():
                if key in page_text.upper():
                    current_team = formal_name
                    break
            page_team_map[page_num] = current_team

    return page_team_map

def parse_with_camelot(pdf_folder="fia_pdfs"):
    all_rows = []

    print("Parsing PDFs using Camelot with Merged-Cell Memory & NLTK Preprocessing...\n")

    for filename in os.listdir(pdf_folder):
        if not filename.endswith(".pdf"):
            continue

        file_path = os.path.join(pdf_folder, filename)
        race_name = extract_race_name(filename)

        print(f"Analyzing: {filename}")

        page_team_map = get_team_per_page(file_path)
        tables = camelot.read_pdf(file_path, pages='all', flavor='lattice')

        for table in tables:
            df = table.df
            if df.empty:
                continue

            team_name = page_team_map.get(table.page, "Unknown Team")
            last_valid_description = ""

            for _, row in df.iterrows():
                row_list = [str(cell).replace('\n', ' ').strip() for cell in row]

                if not any(row_list):
                    continue

                combined_text = " ".join(row_list).upper()
                if "UPDATED COMPONENT" in combined_text or "PRIMARY REASON" in combined_text or "GEOMETRIC DIFFERENCES" in combined_text:
                    continue

                if len(row_list) >= 5:
                    component = row_list[1]
                    reason = row_list[2]
                    geometry = row_list[3]
                    description = row_list[4]
                elif len(row_list) == 4:
                    component = row_list[0]
                    reason = row_list[1]
                    geometry = row_list[2]
                    description = row_list[3]
                else:
                    continue

                if description == "" and component != "":
                    description = last_valid_description
                elif description != "":
                    last_valid_description = description

                if component != "" and len(component) > 2:

                    # --- NLTK PREPROCESSING STEP ---
                    # We clean the text before creating the row dictionary
                    clean_component = clean_text_stopwords(component)
                    clean_reason = clean_text_stopwords(reason)
                    clean_geometry = clean_text_stopwords(geometry)
                    clean_description = description

                    all_rows.append({
                        "Race": race_name,
                        "Team": team_name,
                        "Component": clean_component,
                        "Reason": clean_reason,
                        "Geometry_Changes": clean_geometry,
                        "Description": clean_description
                    })

    if not all_rows:
        return pd.DataFrame()

    final_df = pd.DataFrame(all_rows, columns=["Race", "Team", "Component", "Reason", "Geometry_Changes", "Description"])


    valid_teams = list(CONSTRUCTORS.values())

    final_df = final_df[final_df["Team"].isin(valid_teams)].copy()

    for col in ["Component", "Reason", "Geometry_Changes", "Description"]:
        final_df[col] = final_df[col].str.replace(r'\s+', ' ', regex=True).str.strip()

    return final_df

if __name__ == "__main__":
    df_articles = parse_with_camelot("fia_pdfs")

    if not df_articles.empty:
        print("\n=== PARSING COMPLETE ===")
        print(f"Total structured rows gathered: {len(df_articles)}")
        df_articles.to_csv("f1_upgrades_structured.csv", index=False)
        print("Saved cleanly to 'f1_upgrades_structured.csv'")

        display(df_articles.head(20))
    else:
        print("\n❌ No clean table rows were successfully extracted.")

Parsing PDFs using Camelot with Merged-Cell Memory & NLTK Preprocessing...

Analyzing: Australian_Grand_Prix_doc_7_-_car_presentation_submissions.pdf


Analyzing: Hungarian_Grand_Prix_doc_8_-_car_presentation_submissions.pdf
Analyzing: Abu_Dhabi_Grand_Prix_doc_8_-_car_presentation_submissions.pdf
Analyzing: Japanese_Grand_Prix_doc_9_-_car_presentation_submissions.pdf


/usr/local/lib/python3.12/dist-packages/camelot/parsers/base.py:132: UserWarning: page-5 is image-based, camelot only works on text-based pages.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/camelot/parsers/base.py:132: UserWarning: page-10 is image-based, camelot only works on text-based pages.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/camelot/parsers/base.py:132: UserWarning: page-12 is image-based, camelot only works on text-based pages.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/camelot/parsers/base.py:132: UserWarning: page-14 is image-based, camelot only works on text-based pages.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/camelot/parsers/base.py:132: UserWarning: page-16 is image-based, camelot only works on text-based pages.
  warnings.warn(


Analyzing: Mexico_City_Grand_Prix_doc_7_-_car_presentation_submissions.pdf


Analyzing: United_States_Grand_Prix_doc_10_-_car_presentation_submissions.pdf
Analyzing: Dutch_Grand_Prix_doc_9_-_car_presentation_submissions.pdf
Analyzing: Canadian_Grand_Prix_doc_8_-_car_presentation_submissions.pdf
Analyzing: Emilia_Romagna_Grand_Prix_doc_8_-_car_presentation_submissions.pdf


...

Analyzing: Austrian_Grand_Prix_doc_9_-_car_presentation_submissions.pdf
Analyzing: British_Grand_Prix_doc_9_-_car_presentation_submissions.pdf


Analyzing: Chinese_Grand_Prix_doc_9_-_car_presentation_submissions.pdf
Analyzing: São_Paulo_Grand_Prix_doc_10_-_car_presentation_submissions.pdf
Analyzing: Spanish_Grand_Prix_doc_8_-_car_presentation_submissions.pdf
Analyzing: Qatar_Grand_Prix_doc_8_-_car_presentation_submissions.pdf
Analyzing: Las_Vegas_Grand_Prix_doc_7_-_car_presentation_submissions.pdf
Analyzing: Miami_Grand_Prix_doc_8_-_car_presentation_submissions.pdf
Analyzing: Monaco_Grand_Prix_doc_11_-_car_presentation_submissions.pdf


Analyzing: Italian_Grand_Prix_doc_8_-_car_presentation_submissions.pdf


Analyzing: Singapore_Grand_Prix_doc_10_-_car_presentation_submissions.pdf
Analyzing: Bahrain_Grand_Prix_doc_10_-_car_presentation_submissions.pdf
Analyzing: Azerbaijan_Grand_Prix_doc_8_-_car_presentation_submissions.pdf

=== PARSING COMPLETE ===
Total structured rows gathered: 287
Saved cleanly to 'f1_upgrades_structured.csv'


,Race,Team,Component,Reason,Geometry_Changes,Description
0,Australian Grand Prix,McLaren,Front Corner,Performance - Flow Conditioning,Revised Front Brake Duct Winglet,Revised Front Brake Duct Winglet geometry resu...
1,Australian Grand Prix,McLaren,Front Corner,Circuit specific - Cooling Range,Lower Cooling Front Corner Scoop,Low cooling Front Brake Duct option converting...
2,Australian Grand Prix,McLaren,Beam Wing,Circuit specific - Drag Range,loaded single element Beamwing,In order to improve both aerodynamic efficienc...
3,Australian Grand Prix,McLaren,Beam Wing,Circuit specific - Drag Range,loaded double element Beamwing,In order to improve both aerodynamic efficienc...
4,Australian Grand Prix,Ferrari,Front Suspension,Performance - Flow Conditioning,"Switch pushrod pullrod , upper lower wishbones...",This new front suspension layout has unlocked ...
5,Australian Grand Prix,Ferrari,Sidepod Inlet / Engine cover,Performance - Flow Conditioning,compact sidepod side plan views . Reoptimized ...,Benefiting from an improved onset flow quality...
6,Australian Grand Prix,Ferrari,Rear Wing / Beam Wing,Performance - Local Load,New top rear wing profiles revised lower wing ...,Capitalizing on improved upstream flow energy ...
7,Australian Grand Prix,Red Bull Racing,Front Wing,Performance - Local Load,"First element curls endplate revised , changes...",Redistributed loadings across the elements see...
8,Australian Grand Prix,Red Bull Racing,Nose,Performance - Local Load,Revised fairings element junctions,A consequences of revising the element junctio...
9,Australian Grand Prix,Red Bull Racing,Front Suspension,Performance - Local Load,Suspension fairings revised trackrod wishbones,More local load has been extracted from the re...


In [8]:
display(df_articles.head(100))


,Race,Team,Component,Reason,Geometry_Changes,Description
0,Australian Grand Prix,McLaren,Front Corner,Performance - Flow Conditioning,Revised Front Brake Duct Winglet,Revised Front Brake Duct Winglet geometry resu...
1,Australian Grand Prix,McLaren,Front Corner,Circuit specific - Cooling Range,Lower Cooling Front Corner Scoop,Low cooling Front Brake Duct option converting...
2,Australian Grand Prix,McLaren,Beam Wing,Circuit specific - Drag Range,loaded single element Beamwing,In order to improve both aerodynamic efficienc...
3,Australian Grand Prix,McLaren,Beam Wing,Circuit specific - Drag Range,loaded double element Beamwing,In order to improve both aerodynamic efficienc...
4,Australian Grand Prix,Ferrari,Front Suspension,Performance - Flow Conditioning,"Switch pushrod pullrod , upper lower wishbones...",This new front suspension layout has unlocked ...
...,...,...,...,...,...,...
113,Canadian Grand Prix,Visa Cash App RB,Rear Corner,Performance - Flow Conditioning,shape lower winglet endplate revised .,The vorticity shed from the brake drum winglet...
114,Emilia Romagna Grand Prix,McLaren,Rear Corner,Performance - Flow Conditioning,Revised Rear Corner,Update to multiple Rear Corner and Suspension ...
115,Emilia Romagna Grand Prix,McLaren,Rear Wing,Circuit specific - Drag Range,High Downforce Rear Wing,High Downforce Rear Wing resulting in an effic...
116,Emilia Romagna Grand Prix,McLaren,Beam Wing,Circuit specific - Drag Range,High Downforce Beam Wing,In conjunction with the high downforce Rear Wi...


In [10]:
f1_calendar = {
    "Australian Grand Prix": {"Start": "2025-03-14", "End": "2025-03-16"},
    "Chinese Grand Prix": {"Start": "2025-03-21", "End": "2025-03-23"},
    "Japanese Grand Prix": {"Start": "2025-04-04", "End": "2025-04-06"},
    "Bahrain Grand Prix": {"Start": "2025-04-11", "End": "2025-04-13"},
    "Saudi Arabian Grand Prix": {"Start": "2025-04-18", "End": "2025-04-20"},
    "Miami Grand Prix": {"Start": "2025-05-02", "End": "2025-05-04"},
    "Emilia Romagna Grand Prix": {"Start": "2025-05-16", "End": "2025-05-18"},
    "Monaco Grand Prix": {"Start": "2025-05-23", "End": "2025-05-25"},
    "Spanish Grand Prix": {"Start": "2025-05-30", "End": "2025-06-01"},
    "Canadian Grand Prix": {"Start": "2025-06-13", "End": "2025-06-15"},
    "Austrian Grand Prix": {"Start": "2025-06-27", "End": "2025-06-29"},
    "British Grand Prix": {"Start": "2025-07-04", "End": "2025-07-06"},
    "Belgian Grand Prix": {"Start": "2025-07-25", "End": "2025-07-27"},
    "Hungarian Grand Prix": {"Start": "2025-08-01", "End": "2025-08-03"},
    "Dutch Grand Prix": {"Start": "2025-08-29", "End": "2025-08-31"},
    "Italian Grand Prix": {"Start": "2025-09-05", "End": "2025-09-07"},
    "Azerbaijan Grand Prix": {"Start": "2025-09-19", "End": "2025-09-21"},
    "Singapore Grand Prix": {"Start": "2025-10-03", "End": "2025-10-05"},
    "United States Grand Prix": {"Start": "2025-10-17", "End": "2025-10-19"},
    "Mexico City Grand Prix": {"Start": "2025-10-24", "End": "2025-10-26"},
    "Sao Paulo Grand Prix": {"Start": "2025-11-07", "End": "2025-11-09"},
    "Las Vegas Grand Prix": {"Start": "2025-11-20", "End": "2025-11-22"},
    "Qatar Grand Prix": {"Start": "2025-11-28", "End": "2025-11-30"},
    "Abu Dhabi Grand Prix": {"Start": "2025-12-05", "End": "2025-12-07"}
}

df_articles['Race_Day'] = df_articles['Race'].map(lambda x: f1_calendar.get(x, {}).get("End", "Unknown Date"))
df_articles.to_csv("f1_upgrades_structured.csv", index=False)

In [12]:
import pandas as pd
import pickle
import plotly.express as px
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import KMeans

print("Loading structured F1 data...")
df = pd.read_csv("f1_upgrades_structured.csv")
df.fillna("", inplace=True)

df['Combined_Text'] = (
    df['Component'] + " " +
    df['Reason'] + " " +
    df['Geometry_Changes'] + " " +
    df['Description']
)

print("Running TF-IDF Vectorization...")
tfidf_vectorizer = TfidfVectorizer(max_df=0.85, min_df=2, stop_words='english')
tfidf_matrix = tfidf_vectorizer.fit_transform(df['Combined_Text'])
feature_names = tfidf_vectorizer.get_feature_names_out()

print("Applying PCA (TruncatedSVD) to reduce dimensions...")
pca = TruncatedSVD(n_components=2, random_state=42)
pca_matrix = pca.fit_transform(tfidf_matrix)
df['PCA_X'] = pca_matrix[:, 0]
df['PCA_Y'] = pca_matrix[:, 1]

print("Applying K-Means Clustering...")
NUM_CLUSTERS = 5
kmeans = KMeans(n_clusters=NUM_CLUSTERS, random_state=42, n_init=10)
df['Cluster_ID'] = kmeans.fit_predict(pca_matrix)

# --- 3. APPLY ENGINEERING CATEGORIES ---
print("Mapping mathematical clusters to Engineering Categories...")
cluster_mapping = {
    0: "Cooling & Reliability",
    1: "Floor & Underbody",
    2: "Circuit-Specific Aerodynamics",
    3: "General Aerodynamic Performance",
    4: "Bodywork Flow Conditioning"
}

# Apply the professional labels
df['Engineering_Category'] = df['Cluster_ID'].map(cluster_mapping)

print("\n=== FINAL CATEGORY COUNTS ===")
print(df['Engineering_Category'].value_counts())

# --- 4. VISUALIZATION ---
# Render an interactive scatter plot with the professional labels
fig = px.scatter(
    df,
    x='PCA_X',
    y='PCA_Y',
    color='Engineering_Category',
    hover_data=['Race','Race_Day' ,'Team', 'Component', 'Reason'],
    title='F1 Upgrades: PCA + K-Means Clustering',
    color_discrete_sequence=px.colors.qualitative.Bold
)
fig.update_layout(xaxis_title="Principal Component 1", yaxis_title="Principal Component 2")
fig.show()

print("\n=== SAVING HANDOVER FILES ===")
with open("pca_kmeans_partner_a.pkl", "wb") as f:
    pickle.dump({
        "tfidf_vectorizer": tfidf_vectorizer,
        "pca_model": pca,
        "kmeans_model": kmeans,
        "tfidf_matrix": tfidf_matrix
    }, f)
print("Saved Machine Learning assets to 'pca_kmeans_partner_a.pkl'")

# Save the updated DataFrame with the new X/Y coordinates and engineering labels
df.to_csv("f1_upgrades_kmeans_clustered.csv", index=False)
print("Saved clustered dataset to 'f1_upgrades_kmeans_clustered.csv'")

Loading structured F1 data...
Running TF-IDF Vectorization...
Applying PCA (TruncatedSVD) to reduce dimensions...
Applying K-Means Clustering...
Mapping mathematical clusters to Engineering Categories...

=== FINAL CATEGORY COUNTS ===
Engineering_Category
Circuit-Specific Aerodynamics      65
Bodywork Flow Conditioning         60
Cooling & Reliability              59
Floor & Underbody                  58
General Aerodynamic Performance    45
Name: count, dtype: int64



=== SAVING HANDOVER FILES ===
Saved Machine Learning assets to 'pca_kmeans_partner_a.pkl'
Saved clustered dataset to 'f1_upgrades_kmeans_clustered.csv'


In [14]:
import pandas as pd
import plotly.express as px

df = pd.read_csv("f1_upgrades_kmeans_clustered.csv")

team_counts = df['Team'].value_counts().reset_index()
team_counts.columns = ['Team', 'Total Upgrades']

fig_teams = px.bar(
    team_counts,
    x='Team',
    y='Total Upgrades',
    color='Team',
    title='Total Car Upgrades Brought by Each Team',
    text_auto=True
)
fig_teams.update_layout(xaxis_title="Constructor", yaxis_title="Number of Upgrades")
fig_teams.show()


cluster_counts = df.groupby(['Team', 'Engineering_Category']).size().reset_index(name='Count')

fig_tree = px.treemap(
    cluster_counts,
    path=['Team', 'Engineering_Category'],
    values='Count',
    title='Upgrade Focus Breakdown: Team -> Topic Cluster',
    height=700
)
fig_tree.data[0].textinfo = 'label+text+value'
fig_tree.show()


df['Race_Day'] = pd.to_datetime(df['Race_Day'], errors='coerce')

df = df.sort_values(by='Race_Day')

chronological_races = df['Race'].unique().tolist()

fig_races = px.histogram(
    df,
    x="Race",
    color="Team",
    title="Pace of Development: Upgrades Brought to Each Race",
    barmode="stack",
    hover_data=['Race_Day', 'Component'],
    category_orders={"Race": chronological_races}
)

fig_races.update_layout(xaxis_title="Race Event", yaxis_title="Number of New Components")
fig_races.show()
